# Modeling and Evaluation

This notebook trains, compares, and evaluates machine learning models for late delivery prediction.

The validation set is used for model selection, hyperparameter tuning, and threshold optimization. The test set is kept untouched until the final evaluation.

## 1. Load Prepared Data

In [2]:
import numpy as np
from scipy.sparse import load_npz

# Load processed feature matrices
X_train = load_npz("../artifacts/X_train.npz")
X_validation = load_npz("../artifacts/X_validation.npz")
X_test = load_npz("../artifacts/X_test.npz")

# Load target labels
y_train = np.load("../artifacts/y_train.npy", allow_pickle=True)
y_validation = np.load("../artifacts/y_validation.npy", allow_pickle=True)
y_test = np.load("../artifacts/y_test.npy", allow_pickle=True)

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

print("\nTarget shapes:")
print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

print("\nTarget classes:")
print("Train:", np.unique(y_train))
print("Validation:", np.unique(y_validation))
print("Test:", np.unique(y_test))

Train: (67533, 61)
Validation: (14471, 61)
Test: (14472, 61)

Target shapes:
y_train: (67533,)
y_validation: (14471,)
y_test: (14472,)

Target classes:
Train: ['Late' 'On Time']
Validation: ['Late' 'On Time']
Test: ['Late' 'On Time']


## 2. Baseline Model

In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report

# Baseline model
baseline = DummyClassifier(strategy="most_frequent")

# Train only on training data
baseline.fit(X_train, y_train)

# Evaluate on validation data
y_validation_pred = baseline.predict(X_validation)

baseline_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    y_validation_pred
)

print("Baseline Balanced Accuracy:", baseline_balanced_accuracy)

print("\nBaseline Classification Report:")
print(classification_report(
    y_validation,
    y_validation_pred,
    target_names=["Late", "On Time"],
    zero_division=0
))

Baseline Balanced Accuracy: 0.5

Baseline Classification Report:
              precision    recall  f1-score   support

        Late       0.00      0.00      0.00       773
     On Time       0.95      1.00      0.97     13698

    accuracy                           0.95     14471
   macro avg       0.47      0.50      0.49     14471
weighted avg       0.90      0.95      0.92     14471



## 3. Model Training

In [4]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression model
logistic_model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    solver="liblinear",
    max_iter=2000,
    random_state=42
)

# Train only on the training data
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


## 4. Model Evaluation

In [5]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict on validation set
y_validation_pred_lr = logistic_model.predict(X_validation)

# Balanced Accuracy
lr_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    y_validation_pred_lr
)

print("Logistic Regression Balanced Accuracy:", lr_balanced_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_validation,
    y_validation_pred_lr,
    target_names=["Late", "On Time"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_validation,
    y_validation_pred_lr
))

Logistic Regression Balanced Accuracy: 0.6026687402264748

Classification Report:
              precision    recall  f1-score   support

        Late       0.08      0.53      0.15       773
     On Time       0.96      0.68      0.79     13698

    accuracy                           0.67     14471
   macro avg       0.52      0.60      0.47     14471
weighted avg       0.92      0.67      0.76     14471


Confusion Matrix:
[[ 409  364]
 [4435 9263]]


## 5. XGBoost Model

In [7]:
from xgboost import XGBClassifier

# Encode target labels for XGBoost
y_train_xgb = (y_train == "Late").astype(int)
y_validation_xgb = (y_validation == "Late").astype(int)

# XGBoost model
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Train only on training data
xgb_model.fit(X_train, y_train_xgb)

print("XGBoost trained successfully.")

XGBoost trained successfully.


## 6. XGBoost Evaluation

In [8]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict on validation set
y_validation_pred_xgb = xgb_model.predict(X_validation)

# Convert predictions back to original labels
y_validation_pred_xgb_labels = np.where(
    y_validation_pred_xgb == 1,
    "Late",
    "On Time"
)

# Balanced Accuracy
xgb_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    y_validation_pred_xgb_labels
)

print("XGBoost Balanced Accuracy:", xgb_balanced_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_validation,
    y_validation_pred_xgb_labels,
    target_names=["Late", "On Time"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_validation,
    y_validation_pred_xgb_labels
))

XGBoost Balanced Accuracy: 0.506062489741281

Classification Report:
              precision    recall  f1-score   support

        Late       0.15      0.02      0.03       773
     On Time       0.95      0.99      0.97     13698

    accuracy                           0.94     14471
   macro avg       0.55      0.51      0.50     14471
weighted avg       0.90      0.94      0.92     14471


Confusion Matrix:
[[   14   759]
 [   82 13616]]


## 7. KNN Model

In [9]:
from sklearn.neighbors import KNeighborsClassifier

# KNN model
knn_model = KNeighborsClassifier(
    n_neighbors=15,
    weights="distance",
    n_jobs=-1
)

# Train on the training data
knn_model.fit(X_train, y_train)

print("KNN trained successfully.")

KNN trained successfully.


## 8. KNN Evaluation

In [10]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict on validation set
y_validation_pred_knn = knn_model.predict(X_validation)

# Balanced Accuracy
knn_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    y_validation_pred_knn
)

print("KNN Balanced Accuracy:", knn_balanced_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_validation,
    y_validation_pred_knn,
    target_names=["Late", "On Time"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_validation,
    y_validation_pred_knn
))

KNN Balanced Accuracy: 0.5017273841168493

Classification Report:
              precision    recall  f1-score   support

        Late       0.09      0.01      0.01       773
     On Time       0.95      1.00      0.97     13698

    accuracy                           0.94     14471
   macro avg       0.52      0.50      0.49     14471
weighted avg       0.90      0.94      0.92     14471


Confusion Matrix:
[[    6   767]
 [   59 13639]]


## 9. Random Forest Model

In [11]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train only on the training data
rf_model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


## 10. Random Forest Evaluation

In [12]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict on validation set
y_validation_pred_rf = rf_model.predict(X_validation)

# Balanced Accuracy
rf_balanced_accuracy = balanced_accuracy_score(
    y_validation,
    y_validation_pred_rf
)

print("Random Forest Balanced Accuracy:", rf_balanced_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_validation,
    y_validation_pred_rf,
    target_names=["Late", "On Time"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_validation,
    y_validation_pred_rf
))

Random Forest Balanced Accuracy: 0.5936727526723667

Classification Report:
              precision    recall  f1-score   support

        Late       0.09      0.41      0.15       773
     On Time       0.96      0.78      0.86     13698

    accuracy                           0.76     14471
   macro avg       0.53      0.59      0.51     14471
weighted avg       0.91      0.76      0.82     14471


Confusion Matrix:
[[  314   459]
 [ 2998 10700]]


## 11. Model Comparison

In [13]:
import pandas as pd

# Compare validation performance of all models

model_comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "KNN"
    ],
    "Balanced Accuracy": [
        baseline_balanced_accuracy,
        lr_balanced_accuracy,
        rf_balanced_accuracy,
        xgb_balanced_accuracy,
        knn_balanced_accuracy
    ]
})

print(model_comparison.sort_values(
    by="Balanced Accuracy",
    ascending=False
))

                 Model  Balanced Accuracy
1  Logistic Regression           0.602669
2        Random Forest           0.593673
3              XGBoost           0.506062
4                  KNN           0.501727
0             Baseline           0.500000


Based on the validation results, **Logistic Regression is currently the best-performing model in terms of Balanced Accuracy**.

However, the main objective of this project is to identify orders that are likely to be **Late**. Therefore, Balanced Accuracy alone is not sufficient to determine the most useful model configuration. We will next perform **Threshold Optimization** for the Logistic Regression model to investigate whether lowering the classification threshold can improve the **Recall of the Late class** while maintaining an acceptable balance between Recall, Precision, F1-score, and Balanced Accuracy.


## Evaluation Metrics

### Accuracy
Accuracy measures the overall proportion of orders that are correctly classified by the model as either **Late** or **On Time**.
 
يعني من جميع الطلبات، كم طلبًا صنّفه النموذج في الفئة الصحيحة، سواء كان **Late** أو **On Time**.

### Recall (Late Class)
Recall measures the proportion of orders that were **actually Late** and were correctly identified as Late by the model. In this project, Recall is particularly important because the main objective is to detect potentially delayed orders.
 
يعني من جميع الطلبات التي كانت **متأخرة فعلًا**، كم طلبًا استطاع النموذج اكتشافه وتصنيفه على أنه **Late**. وهذا مهم لأن هدفنا الأساسي هو تقليل عدد الطلبات المتأخرة التي يفشل النموذج في اكتشافها.

### Precision (Late Class)
Precision measures the proportion of orders predicted by the model as **Late** that were actually Late. It indicates how reliable the model's late-delivery alerts are.
 
يعني من جميع الطلبات التي قال النموذج عنها **Late**، كم طلبًا كان متأخرًا فعلًا. وكلما ارتفعت Precision، كانت إنذارات النموذج أكثر موثوقية وأقل تسببًا في الإنذارات الكاذبة.

### F1-Score (Late Class)
The F1-score provides a balance between **Precision and Recall** for the Late class. It evaluates whether the model can detect delayed orders while maintaining a reasonable level of prediction reliability.
 
يعني مقياس يوازن بين **Recall وPrecision**؛ فلا نريد فقط اكتشاف أكبر عدد ممكن من الطلبات المتأخرة، وإنما نريد أيضًا أن تكون الإنذارات التي يعطيها النموذج صحيحة بدرجة معقولة.

### Balanced Accuracy
Balanced Accuracy measures the average classification performance across the **Late** and **On Time** classes, giving equal importance to both classes. This is appropriate because the dataset contains substantially more On Time orders than Late orders.
  
يعني يقيس أداء النموذج في الفئتين **Late وOn Time** بشكل متوازن، بحيث لا تؤدي كثرة حالات **On Time** إلى إعطاء انطباع مبالغ فيه عن أداء النموذج.

### Evaluation Priority
The primary metric for this project is **Recall for the Late class**, because the main objective is to identify as many potentially delayed orders as possible.

**Balanced Accuracy** is used as the main overall comparison metric between models, while **Precision** and **F1-score** are monitored to ensure that increasing Late detection does not result in an excessive number of false alerts.

Accuracy is reported as a supplementary metric and is not used alone for model selection because of the class imbalance.


## 12. Threshold Optimization`

In [15]:
import numpy as np
from sklearn.metrics import (
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Get probability scores for the Late class
y_validation_proba_lr = logistic_model.predict_proba(X_validation)

# Identify the column corresponding to "Late"
late_class_index = list(logistic_model.classes_).index("Late")

late_probabilities = y_validation_proba_lr[:, late_class_index]

# Thresholds to evaluate
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]

threshold_results = []

for threshold in thresholds:

    y_pred_threshold = np.where(
        late_probabilities >= threshold,
        "Late",
        "On Time"
    )

    threshold_results.append({
        "Threshold": threshold,
        "Balanced Accuracy": balanced_accuracy_score(
            y_validation,
            y_pred_threshold
        ),
        "Late Precision": precision_score(
            y_validation,
            y_pred_threshold,
            pos_label="Late",
            zero_division=0
        ),
        "Late Recall": recall_score(
            y_validation,
            y_pred_threshold,
            pos_label="Late",
            zero_division=0
        ),
        "Late F1": f1_score(
            y_validation,
            y_pred_threshold,
            pos_label="Late",
            zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

print(threshold_results)

   Threshold  Balanced Accuracy  Late Precision  Late Recall   Late F1
0       0.50           0.602669        0.084434     0.529107  0.145629
1       0.45           0.601696        0.078561     0.601552  0.138972
2       0.40           0.598367        0.072448     0.708926  0.131462
3       0.35           0.541745        0.058392     0.927555  0.109868
4       0.30           0.518526        0.055432     0.965071  0.104842
5       0.25           0.512152        0.054699     0.981889  0.103625
6       0.20           0.503770        0.053802     0.998706  0.102103


## 13. Select Final Model

Based on the validation results, Logistic Regression achieved the highest Balanced Accuracy among the evaluated models.

Threshold optimization showed that lowering the decision threshold substantially increased the Recall of the Late class, but this was accompanied by a decrease in Precision, F1-score, and Balanced Accuracy.

Therefore, a threshold of **0.50** is selected as the final operating threshold because it provides the best Balanced Accuracy and the highest Late-class F1-score among the evaluated thresholds.

In [16]:
# Select the final model and operating threshold

final_model = logistic_model
final_threshold = 0.50

print("Final model: Logistic Regression")
print("Final threshold:", final_threshold)

Final model: Logistic Regression
Final threshold: 0.5


## 14. Final Test Evaluation

The selected Logistic Regression model is evaluated once on the previously untouched test set using the selected classification threshold of 0.50.

The test set is used only to provide an unbiased estimate of the final model's generalization performance.

In [17]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Predict probabilities for the Late class
y_test_proba = final_model.predict_proba(X_test)

# Find the probability column corresponding to Late
late_class_index = list(final_model.classes_).index("Late")

late_probabilities_test = y_test_proba[:, late_class_index]

# Apply the selected threshold
y_test_pred = np.where(
    late_probabilities_test >= final_threshold,
    "Late",
    "On Time"
)

# Final evaluation
test_accuracy = accuracy_score(y_test, y_test_pred)
test_balanced_accuracy = balanced_accuracy_score(y_test, y_test_pred)

print("Final Test Accuracy:", test_accuracy)
print("Final Test Balanced Accuracy:", test_balanced_accuracy)

print("\nFinal Test Classification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Late", "On Time"],
    zero_division=0
))

print("\nFinal Test Confusion Matrix:")
print(confusion_matrix(
    y_test,
    y_test_pred
))

Final Test Accuracy: 0.6519485903814262
Final Test Balanced Accuracy: 0.4650839598866695

Final Test Classification Report:
              precision    recall  f1-score   support

        Late       0.05      0.25      0.09       957
     On Time       0.93      0.68      0.79     13515

    accuracy                           0.65     14472
   macro avg       0.49      0.47      0.44     14472
weighted avg       0.87      0.65      0.74     14472


Final Test Confusion Matrix:
[[ 239  718]
 [4319 9196]]


## 15. Feature Interpretation

The final Logistic Regression model is examined to understand the contribution of the transformed features to the prediction of late deliveries.

Because categorical variables were one-hot encoded during preprocessing, feature interpretation is performed using the coefficients of the transformed feature space.

In [19]:
import joblib
import numpy as np
import pandas as pd

# Load the fitted preprocessor from Notebook 5
preprocessor = joblib.load("../artifacts/preprocessor.joblib")

# Extract transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Extract Logistic Regression coefficients
coefficients = final_model.coef_[0]

# Create feature importance table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Coefficient": np.abs(coefficients)
})

# Sort by absolute coefficient
top_features = feature_importance.sort_values(
    by="Absolute Coefficient",
    ascending=False
).head(15)

print("Top 15 most influential transformed features:")
print(top_features.to_string(index=False))

Top 15 most influential transformed features:
               Feature  Coefficient  Absolute Coefficient
  cat__seller_state_MA    -2.004148              2.004148
  cat__seller_state_RO     1.427436              1.427436
cat__customer_state_RO     1.170938              1.170938
  cat__seller_state_AM    -1.117058              1.117058
cat__customer_state_AL    -1.081046              1.081046
  cat__seller_state_MS     0.989296              0.989296
cat__customer_state_AC     0.913656              0.913656
cat__customer_state_AM     0.884292              0.884292
  cat__seller_state_PI     0.866074              0.866074
  cat__seller_state_SE     0.805820              0.805820
  cat__seller_state_SP    -0.798672              0.798672
cat__customer_state_MA    -0.760510              0.760510
  cat__seller_state_RJ    -0.691928              0.691928
cat__customer_state_PR     0.659389              0.659389
  cat__seller_state_RS     0.654052              0.654052


## 16. Save Final Model and Results

The selected Logistic Regression model, preprocessing pipeline, final classification threshold, and evaluation results are saved as project artifacts for reproducibility and future use.

In [20]:
import joblib
import json
import os

# Save the final model
joblib.dump(
    final_model,
    "../artifacts/final_logistic_model.joblib"
)

# Save the final threshold
with open("../artifacts/final_threshold.json", "w", encoding="utf-8") as f:
    json.dump(
        {"threshold": float(final_threshold)},
        f,
        indent=4
    )

# Save final test results
final_results = {
    "model": "Logistic Regression",
    "threshold": float(final_threshold),
    "test_accuracy": float(test_accuracy),
    "test_balanced_accuracy": float(test_balanced_accuracy)
}

with open("../artifacts/final_results.json", "w", encoding="utf-8") as f:
    json.dump(
        final_results,
        f,
        indent=4
    )

print("Final model and evaluation artifacts saved successfully.")

Final model and evaluation artifacts saved successfully.


## Experiment 1 – Baseline Modeling Summary

Five classification approaches were evaluated using the leakage-safe feature set: Baseline, Logistic Regression, KNN, Random Forest, and XGBoost.

Logistic Regression achieved the highest validation Balanced Accuracy (0.603) and a Late-class Recall of approximately 0.53, making it the best-performing model among the evaluated approaches on the validation set.

Threshold optimization showed that lowering the threshold increased Late Recall but reduced Precision, F1-score, and Balanced Accuracy. Therefore, a threshold of 0.50 was retained for the final evaluation.

When evaluated once on the untouched test set, the final model achieved an Accuracy of 65.2%, a Balanced Accuracy of 46.5%, a Late Recall of 25.0%, and a Late F1-score of 9.2%.

The difference between validation and test performance indicates limited generalization of the current feature set. The model was therefore retained as the baseline experiment, providing a reference point for further feature-engineering improvements.

### Model Selection Notes

- **Logistic Regression:** Selected as the best baseline model based on validation performance.
- **Random Forest:** Performed close to Logistic Regression but achieved lower Balanced Accuracy and Late Recall.
- **XGBoost:** Showed weak performance for identifying Late orders with the selected configuration.
- **KNN:** Performed close to the majority-class baseline and was therefore not competitive.
- **Baseline:** Used only as a reference point.